# Generating the Paper Tables

This notebook builds **all tables** in the paper from trained checkpoints,
baseline CSVs, and caches from the other notebooks. No table value is written
by hand.

### Inputs

| Source | Produced by | Used in table |
| --- | --- | --- |
| checkpoints (`*.pth`) | training notebooks | benchmark, quantization |
| `results/baselines_*.csv` | `notebooks/01_baselines.ipynb` | benchmark, CPU latency |
| `results/inference_profile.csv` | `notebooks/04_inference_benchmark.ipynb` | efficiency, CPU latency |
| `results/ablation_synthetic.csv` | `notebooks/06_ablations.ipynb` | ablation |
| `results/theorem1_bias_shifting.csv` | `notebooks/06_ablations.ipynb` | Proposition 1 |
| `results/predictions/baselines_*.npz` | `notebooks/01_baselines.ipynb` | statistical reliability |

### Outputs

Each table is written to `results/tables/<ad>.tex` and printed here as LaTeX.
The copies under `paper/neurocomputing/generated/<ad>.tex` are derived from
these canonical files by `paper/neurocomputing/prepare_tables.py`; the paper
includes them with `\input{generated/<ad>.tex}`, and table values are never
edited by hand.

> **Note (test-observed operating point):** Quantization fidelity is measured on
> `split.loader("test")`, and the budget `(Λ_max, τ_tol)` was chosen against this
> metric. Single-split fidelity values are therefore a reported operating point,
> not a held-out estimate; the paper states this in §4.1.2 and §4.8. The budget
> enters no accuracy number and affects only Table 10.

> **Run order:** training → `01`–`02` → `03`–`06` figure notebooks → this notebook.

In [1]:
import json
import sys
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# Find the project root regardless of the notebook's working directory.
cwd = Path.cwd().resolve()
project_root = next(
    (path for path in [cwd, *cwd.parents] if (path / "utils" / "experiments.py").exists()),
    cwd.parent,
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.reproducibility import set_seed
from utils.experiments import (
    DATASETS,
    RESULTS_DIR,
    baseline_path,
    compact_model,
    load_checkpoint,
    load_splits,
)
from utils.analysis import evaluate_scorecard_quantization_analysis

set_seed(42)

TABLE_DIR = RESULTS_DIR / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
# The paper source reads from this directory via \input; tables are versioned.

# Quantization budgets come from a single source (utils/experiments.py).
# They differ per dataset/variant; no second copy is kept here.

ORDER = ["COMPAS", "German Credit", "UCI Adult", "Synthetic"]
KEYS = ["compas", "german", "adult", "synthetic"]

# tab:scorecard_fidelity also reports the COMPAS variant with protected
# attributes removed, in addition to the main benchmark. This key enters only
# quantization_fidelity.csv; scorenet_metrics.csv stays limited to the main
# benchmark (Tables 2-3 and the rule budget ranges depend on it).
QUANT_KEYS = KEYS + ["compas_noprot"]
QUANT_LABEL = {"compas_noprot": r"COMPAS (no sex/race)"}

print(f"Project root : {project_root}")
print(f"Table directory : {TABLE_DIR}")


def write_table(name, lines, caption=None):
    """Write the LaTeX body to file and print it."""
    text = "\n".join(line for line in lines if line is not None)
    path = TABLE_DIR / f"{name}.tex"
    path.write_text(text + "\n", encoding="utf-8")
    print("=" * 100)
    print(f"% {name}" + (f" — {caption}" if caption else ""))
    print("=" * 100)
    print(text)
    print()
    return path


def require_artifact(path, produced_by):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found.\nRun this notebook first: {produced_by}"
        )
    return path

Project root : /home/ahmet/Projects/ScoreNetSingle
Table directory : /home/ahmet/Projects/ScoreNetSingle/results/tables


In [2]:
# ==============================================================================
# 1. SCORE-NET METRICS AND QUANTIZATION (FROM CHECKPOINTS)
# ==============================================================================
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score


def classification_metrics(model, X, y):
    """Accuracy, macro/weighted F1, and ROC-AUC (binary: positive class; multiclass: OvR)."""
    with torch.no_grad():
        probs = torch.softmax(model(X), dim=1).cpu().numpy()
    preds = probs.argmax(axis=1)
    y_true = y.cpu().numpy()

    scores = {
        "Acc": accuracy_score(y_true, preds) * 100.0,
        "F1": f1_score(y_true, preds, average="macro") * 100.0,
        "F1_Weighted": f1_score(y_true, preds, average="weighted") * 100.0,
    }
    try:
        if probs.shape[1] == 2:
            scores["AUC"] = roc_auc_score(y_true, probs[:, 1]) * 100.0
        else:
            scores["AUC"] = roc_auc_score(
                y_true, probs, multi_class="ovr", average="macro"
            ) * 100.0
    except ValueError:
        scores["AUC"] = float("nan")
    return scores


scorenet_rows, quant_rows, splits = [], [], {}

for key in QUANT_KEYS:
    spec = DATASETS[key]
    split = load_splits(key)
    splits[key] = split
    train_loader = split.loader("train")

    for variant, checkpoint in [
        ("Global Best", spec.global_checkpoint),
        ("Compact", spec.final_checkpoint),
    ]:
        raw = load_checkpoint(checkpoint)
        H_init = int(raw.H)
        model = compact_model(raw, train_loader)

        metrics = classification_metrics(model, split.X_test, split.y_test)
        W_f = model.feature_linear.weight.detach().cpu().numpy()
        W_o = model.output_linear.weight.detach().cpu().numpy()
        active_links = int((np.abs(W_f) > 1e-4).sum())
        active_out = int((np.abs(W_o) > 1e-4).sum())

        scorenet_row = {
            "Dataset": spec.display_name,
            "Key": key,
            "Variant": variant,
            "H_init": H_init,
            "H": int(model.H),
            "Avg_Rule_Length": active_links / max(model.H, 1),
            "Sparsity_%": (1.0 - active_links / (H_init * split.n_features)) * 100.0,
            "Active_Params": active_links + 2 * model.H + active_out + model.C,
            **metrics,
        }
        if key in KEYS:
            scorenet_rows.append(scorenet_row)

        # Quantization: only W_f is mapped to integers; W_o stays continuous.
        budget = spec.quantization(variant)
        _, df_adapt, df_rule = evaluate_scorecard_quantization_analysis(
            model,
            split.loader("test", batch_size=512),
            tolerance_error=budget.tolerance,
            max_adaptive_scale=budget.max_scale,
        )
        # Float, Scorecard, and Delta all come from the SAME evaluation run that
        # measures fidelity. Recomputing float accuracy (scorecard + delta)
        # introduces floating-point noise and can print the same model's
        # accuracy 0.01 points apart in Tables 2/3/7 and Table 4. The value is
        # therefore taken directly from the "Original Float Model" row of the
        # quantization analysis and checked against the metric measured above.
        float_row = df_adapt[df_adapt["Method"] == "Original Float Model"].iloc[0]
        adaptive = df_adapt[df_adapt["Method"] == "Rule-Based Adaptive Quantization"].iloc[0]

        float_acc = float(float_row["Accuracy (%)"])
        scorecard_acc = float(adaptive["Accuracy (%)"])
        if not np.isclose(float_acc, metrics["Acc"], rtol=0.0, atol=1e-9):
            raise ValueError(
                f"{key}/{variant}: quantization-analysis float accuracy "
                f"({float_acc!r}) does not match classification_metrics ({metrics['Acc']!r}). "
                "The two evaluation paths may not see the same model and the same "
                "test set."
            )

        quant_rows.append({
            "Dataset": spec.display_name,
            "Key": key,
            "Variant": variant,
            "H": int(model.H),
            "Float_Acc_%": float_acc,
            "Scorecard_Acc_%": scorecard_acc,
            "Delta_Acc_%": scorecard_acc - float_acc,
            "Fidelity_%": float(adaptive["Fidelity (%)"]),
            "Max_Logit_Drift": float(adaptive["Max Logit Drift"]),
            "Mean_Lambda": float(adaptive["Mean Scale (λ)"]),
            "Integer_Min": int(adaptive["Min Integer Score"]),
            "Integer_Max": int(adaptive["Max Integer Score"]),
            "Max_Scale": int(budget.max_scale),
            "Tolerance": float(budget.tolerance),
            "Lambda_Values": ", ".join(
                str(int(v)) for v in sorted(df_rule["Optimal_Scale_Factor"].unique())
            ),
        })

df_scorenet = pd.DataFrame(scorenet_rows)
df_quant = pd.DataFrame(quant_rows)
df_scorenet.to_csv(RESULTS_DIR / "scorenet_metrics.csv", index=False)
df_quant.to_csv(RESULTS_DIR / "quantization_fidelity.csv", index=False)

print(df_scorenet[
    ["Dataset", "Variant", "H_init", "H", "Acc", "F1", "AUC", "Active_Params"]
].to_string(index=False))
print()
print(df_quant[
    ["Dataset", "Variant", "Float_Acc_%", "Scorecard_Acc_%", "Delta_Acc_%",
     "Fidelity_%", "Mean_Lambda", "Integer_Min", "Integer_Max"]
].to_string(index=False))

[Data] Loading German Credit Risk...
-> German Credit ready: train shape: (800, 48), Test Shape: (100, 48), feature count: 48, Val Shape: (100, 48)
[Data] Loading UCI Adult Census Income...


-> UCI Adult ready: train shape: (32559, 97), Test Shape: (9045, 97), feature count: 97, Val Shape: (3618, 97)


      Dataset     Variant  H_init   H       Acc        F1       AUC  Active_Params
       COMPAS Global Best      32  11 68.329868 67.806422 72.820254             60
       COMPAS     Compact      32   5 66.320166 65.706699 70.920652             28
German Credit Global Best      64  64 76.000000 68.000000 70.714286           2224
German Credit     Compact      64  46 75.000000 67.911693 75.047619            310
    UCI Adult Global Best      64  51 85.295744 79.148909 91.283313            556
    UCI Adult     Compact      64  43 85.605307 79.354346 91.180699            256
    Synthetic Global Best    1024 788 95.775000 95.768728 99.542089          24407
    Synthetic     Compact    1024 565 90.025000 90.017879 99.344685          15199

                         Dataset     Variant  Float_Acc_%  Scorecard_Acc_%  Delta_Acc_%  Fidelity_%  Mean_Lambda  Integer_Min  Integer_Max
                          COMPAS Global Best    68.329868        68.399168     0.069300   99.792099     6.272727 

In [3]:
# ==============================================================================
# 2. TABLE 1 — BENCHMARK DATASET SUMMARY
# ==============================================================================
DOMAINS = {
    "COMPAS": "Criminal Justice",
    "German Credit": "Credit Underwriting",
    "UCI Adult": "Socio-Economic",
    "Synthetic": "Multiclass Benchmark",
}

summary_rows = []
for key in KEYS:
    split = splits[key]
    summary_rows.append({
        "Dataset": DATASETS[key].display_name,
        "M": split.n_train + split.n_val + split.n_test,
        "N": split.n_features,
        "C": split.n_classes,
        "Train": split.n_train,
        "Val": split.n_val,
        "Test": split.n_test,
    })
df_summary = pd.DataFrame(summary_rows).set_index("Dataset").loc[
    [DATASETS[k].display_name for k in KEYS]
].reset_index()

# Protocol notes come from a single source (DatasetSpec.protocol_note).
protocol_summary = "; ".join(
    f"{DATASETS[k].display_name}: {DATASETS[k].protocol_note}" for k in KEYS
)

lines = [
    r"\begin{table*}[t]",
    r"\centering",
    rf"\caption{{Summary of benchmark datasets. Train/validation/test counts follow "
    rf"the stratified protocol registered for each dataset ({protocol_summary}).}}",
    r"\label{tab:dataset_summary}",
    r"\renewcommand{\arraystretch}{1.2}",
    r"\resizebox{\textwidth}{!}{%",
    r"\begin{tabular}{l ccccc l}",
    r"\toprule",
    r"\textbf{Dataset} & \textbf{Samples ($M$)} & \textbf{Features ($N$)} & "
    r"\textbf{Classes ($C$)} & \textbf{Train / Val / Test} & \textbf{Test Share} & "
    r"\textbf{Domain} \\",
    r"\midrule",
]
for _, row in df_summary.iterrows():
    lines.append(
        f"{row['Dataset']} & {row['M']:,} & {row['N']} & {row['C']} & "
        f"{row['Train']:,} / {row['Val']:,} / {row['Test']:,} & "
        f"{row['Test'] / row['M'] * 100:.0f}\\% & {DOMAINS[row['Dataset']]} \\\\"
    )
lines += [r"\bottomrule", r"\end{tabular}%", r"}", r"\end{table*}"]
write_table("tab_dataset_summary", lines, "dataset overview")

% tab_dataset_summary — dataset overview
\begin{table*}[t]
\centering
\caption{Summary of benchmark datasets. Train/validation/test counts follow the stratified protocol registered for each dataset (COMPAS: 72/8/20; German Credit: 80/10/10; UCI Adult: 72/8/20; Synthetic: 72/8/20).}
\label{tab:dataset_summary}
\renewcommand{\arraystretch}{1.2}
\resizebox{\textwidth}{!}{%
\begin{tabular}{l ccccc l}
\toprule
\textbf{Dataset} & \textbf{Samples ($M$)} & \textbf{Features ($N$)} & \textbf{Classes ($C$)} & \textbf{Train / Val / Test} & \textbf{Test Share} & \textbf{Domain} \\
\midrule
COMPAS & 7,214 & 13 & 2 & 5,193 / 578 / 1,443 & 20\% & Criminal Justice \\
German Credit & 1,000 & 48 & 2 & 800 / 100 / 100 & 10\% & Credit Underwriting \\
UCI Adult & 45,222 & 97 & 2 & 32,559 / 3,618 / 9,045 & 20\% & Socio-Economic \\
Synthetic & 20,000 & 30 & 20 & 14,400 / 1,600 / 4,000 & 20\% & Multiclass Benchmark \\
\bottomrule
\end{tabular}%
}
\end{table*}



PosixPath('/home/ahmet/Projects/ScoreNetSingle/results/tables/tab_dataset_summary.tex')

In [4]:
# ==============================================================================
# 3. TABLE 2 — COMPREHENSIVE BENCHMARK COMPARISON
# ==============================================================================
# Convert all values into a long table normalized to percentages.
families = {
    "Interpretable": "Inherently Interpretable",
    "SOTA Ensemble": "SOTA Tree Ensembles",
    "Geometric & DL": "Geometric & Deep Models",
}
records = []

for key in KEYS:
    path = baseline_path(key)
    if not path.exists():
        print(f"[WARNING] {path} is missing; only SCORE-Net rows will be written for this dataset.")
        continue
    df_b = pd.read_csv(path)
    for _, row in df_b.iterrows():
        records.append({
            "Key": key,
            "Family": row["Category"],
            "Model": row["Model"],
            "Acc": float(row["Accuracy"]) * 100.0,
            "F1": float(row["F1_Macro"]) * 100.0,
            "AUC": float(row["ROC_AUC"]) * 100.0,
        })

for _, row in df_scorenet.iterrows():
    records.append({
        "Key": row["Key"],
        "Family": "Geometric & DL",
        "Model": (
            r"\modelname{} (Global Best)" if row["Variant"] == "Global Best"
            else r"\modelname{} (Compact)"
        ),
        "Acc": row["Acc"],
        "F1": row["F1"],
        "AUC": row["AUC"],
    })

df_bench = pd.DataFrame(records)
df_bench.to_csv(RESULTS_DIR / "benchmark_comparison.csv", index=False)


def fmt(value, bold=False, underline=False):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return "--"
    text = f"{value:.2f}"
    if bold:
        text = rf"\textbf{{{text}}}"
    if underline:
        text = rf"\underline{{{text}}}"
    return text


def rank_flags(frame, metric):
    """(best, second) values of a metric inside a subset."""
    values = sorted({round(v, 6) for v in frame[metric].dropna()}, reverse=True)
    return (values[0] if values else None, values[1] if len(values) > 1 else None)


bench_keys = [k for k in KEYS if k in set(df_bench["Key"])]
colspec = "ll" + " ccc" * len(bench_keys)
dataset_headers = " & ".join(
    rf"\multicolumn{{3}}{{c}}{{\textbf{{{DATASETS[k].display_name}}}}}"
    for k in bench_keys
)
cmidrules = " ".join(
    rf"\cmidrule(lr){{{3 + 3 * i}-{5 + 3 * i}}}" for i in range(len(bench_keys))
)

lines = [
    r"\begin{table*}[t]",
    r"\centering",
    r"\caption{Test accuracy, macro-F1 and ROC-AUC (\%) against 13 baselines. Best per family "
    r"in bold, best per dataset underlined.}",
    r"\label{tab:benchmark_comparison}",
    r"\small",
    r"\setlength{\tabcolsep}{2.0pt}",
    r"\renewcommand{\arraystretch}{1.12}",
    rf"\begin{{tabular}}{{{colspec}}}",
    r"\toprule",
    r"\multirow{2}{*}{\textbf{Family}} & \multirow{2}{*}{\textbf{Model}} & "
    + dataset_headers + r" \\",
    cmidrules,
    r" & & " + " & ".join([r"\textbf{Acc} & \textbf{F1} & \textbf{AUC}"] * len(bench_keys)) + r" \\",
    r"\midrule",
]

for family, family_title in families.items():
    subset = df_bench[df_bench["Family"] == family]
    if subset.empty:
        continue

    models = list(dict.fromkeys(subset["Model"]))
    title = family_title.replace("&", r"\&")
    words = title.split(" ")
    if len(words) > 2:
        title = " ".join(words[:2]) + r"}\\\textbf{" + " ".join(words[2:])
    family_label = (
        r"\multirow{" + str(len(models)) + r"}{*}{\shortstack[l]{\textbf{"
        + title + r"}}}"
    )

    overall = {}
    for key in bench_keys:
        per_key = df_bench[df_bench["Key"] == key]
        overall[key] = {metric: rank_flags(per_key, metric) for metric in ("Acc", "F1", "AUC")}

    for index, model_name in enumerate(models):
        cells = []
        for key in bench_keys:
            row = subset[(subset["Key"] == key) & (subset["Model"] == model_name)]
            if row.empty:
                cells += ["--", "--", "--"]
                continue
            row = row.iloc[0]
            for metric in ("Acc", "F1", "AUC"):
                best, second = rank_flags(subset[subset["Key"] == key], metric)
                value = row[metric]
                global_best, global_second = overall[key][metric]
                cells.append(
                    fmt(
                        value,
                        bold=(best is not None and abs(value - best) < 5e-3),
                        underline=(global_best is not None and abs(value - global_best) < 5e-3),
                    )
                )
        prefix = family_label if index == 0 else ""
        lines.append(f"{prefix} & {model_name} & " + " & ".join(cells) + r" \\")

    if family != list(families)[-1]:
        lines.append(r"\midrule")

lines += [r"\bottomrule", r"\end{tabular}", r"\end{table*}"]
write_table("tab_benchmark_comparison", lines, "main benchmark")

% tab_benchmark_comparison — main benchmark
\begin{table*}[t]
\centering
\caption{Test accuracy, macro-F1 and ROC-AUC (\%) against 13 baselines. Best per family in bold, best per dataset underlined.}
\label{tab:benchmark_comparison}
\small
\setlength{\tabcolsep}{2.0pt}
\renewcommand{\arraystretch}{1.12}
\begin{tabular}{ll ccc ccc ccc ccc}
\toprule
\multirow{2}{*}{\textbf{Family}} & \multirow{2}{*}{\textbf{Model}} & \multicolumn{3}{c}{\textbf{COMPAS}} & \multicolumn{3}{c}{\textbf{German Credit}} & \multicolumn{3}{c}{\textbf{UCI Adult}} & \multicolumn{3}{c}{\textbf{Synthetic}} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-8} \cmidrule(lr){9-11} \cmidrule(lr){12-14}
 & & \textbf{Acc} & \textbf{F1} & \textbf{AUC} & \textbf{Acc} & \textbf{F1} & \textbf{AUC} & \textbf{Acc} & \textbf{F1} & \textbf{AUC} & \textbf{Acc} & \textbf{F1} & \textbf{AUC} \\
\midrule
\multirow{4}{*}{\shortstack[l]{\textbf{Inherently Interpretable}}} & Logistic Regression & \underline{\textbf{68.75}} & 67.13 & \textbf{73.22} & 

PosixPath('/home/ahmet/Projects/ScoreNetSingle/results/tables/tab_benchmark_comparison.tex')

In [5]:
# ==============================================================================
# 4. TABLE 3 — COMPUTATIONAL RESOURCE PROFILE (CACHE FROM NOTEBOOK 2)
# ==============================================================================
profile_path = require_artifact(
    RESULTS_DIR / "inference_profile.csv",
    "notebooks/04_inference_benchmark.ipynb",
)
df_profile = pd.read_csv(profile_path)

meta_path = RESULTS_DIR / "inference_benchmark_meta.json"
meta = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.exists() else {}


def pm(mean, std, decimals=2):
    if pd.isna(mean):
        return "N/A"
    if pd.isna(std):
        return f"{mean:.{decimals}f}"
    return f"{mean:.{decimals}f}$\\pm${std:.{decimals}f}"


lines = [
    r"\begin{table*}[t]",
    r"\centering",
    r"\caption{Computational resource profile and dual-device inference latency "
    r"of \modelname{}. Latencies are means $\pm$ standard deviations over "
    rf"$N={meta.get('n_batch_runs', '?')}$ batch and "
    rf"$N={meta.get('n_single_runs', '?')}$ single-query runs after "
    rf"{meta.get('n_warmup', '?')} warmups; batch time divided by test-set size "
    r"gives the amortized per-instance cost compared with the baselines in "
    r"Supplementary Table~S4.}",
    r"\label{tab:computational_efficiency}",
    r"\setlength{\tabcolsep}{2pt}",
    r"\begin{tabular}{ll c c ccc c cc}",
    r"\toprule",
    r"\textbf{Dataset} & \textbf{Variant} & \textbf{Test $N$} & \textbf{Acc (\%)} & "
    r"\textbf{Rules} & \textbf{Avg Len} & \textbf{Sparsity (\%)} & "
    r"\textbf{Active params} & \textbf{CPU [ms\,/\,$\mu$s]} & \textbf{GPU [ms\,/\,$\mu$s]} \\",
    r"\midrule",
]

current_dataset = None
for _, row in df_profile.iterrows():
    if current_dataset is not None and row["Dataset"] != current_dataset:
        lines.append(r"\midrule")
    current_dataset = row["Dataset"]

    variant = "Global Best" if "Global" in row["Variant"] else "Compact"
    lines.append(
        f"{row['Dataset']} & {variant} & {int(row['Test_N']):,} & "
        f"{row['Accuracy_%']:.2f} & {int(row['H_active'])}/{int(row['H_init'])} & "
        f"{row['Avg_Rule_Length']:.2f} & {row['Overall_Sparsity_%']:.1f}\\% & "
        f"{int(row['Active_Params']):,} & "
        f"{pm(row['CPU_Batch_ms'], row['CPU_Batch_std_ms'])} / "
        f"{pm(row['CPU_Single_us'], row['CPU_Single_std_us'], 1)} & "
        f"{pm(row['GPU_Batch_ms'], row['GPU_Batch_std_ms'])} / "
        f"{pm(row['GPU_Single_us'], row['GPU_Single_std_us'], 1)} \\\\"
    )

lines += [r"\bottomrule", r"\end{tabular}", r"\end{table*}"]
write_table("tab_computational_efficiency", lines, "resource profile")

% tab_computational_efficiency — resource profile
\begin{table*}[t]
\centering
\caption{Computational resource profile and dual-device inference latency of \modelname{}. Latencies are means $\pm$ standard deviations over $N=100$ batch and $N=500$ single-query runs after 50 warmups; batch time divided by test-set size gives the amortized per-instance cost compared with the baselines in Supplementary Table~S4.}
\label{tab:computational_efficiency}
\setlength{\tabcolsep}{2pt}
\begin{tabular}{ll c c ccc c cc}
\toprule
\textbf{Dataset} & \textbf{Variant} & \textbf{Test $N$} & \textbf{Acc (\%)} & \textbf{Rules} & \textbf{Avg Len} & \textbf{Sparsity (\%)} & \textbf{Active params} & \textbf{CPU [ms\,/\,$\mu$s]} & \textbf{GPU [ms\,/\,$\mu$s]} \\
\midrule
COMPAS & Global Best & 1,443 & 68.33 & 11/32 & 1.27 & 96.6\% & 60 & 0.13$\pm$0.00 / 63.3$\pm$8.3 & 0.36$\pm$0.01 / 354.6$\pm$11.6 \\
COMPAS & Compact & 1,443 & 66.32 & 5/32 & 1.20 & 98.6\% & 28 & 0.12$\pm$0.00 / 62.5$\pm$8.4 & 0.36$\pm$0.01 / 3

PosixPath('/home/ahmet/Projects/ScoreNetSingle/results/tables/tab_computational_efficiency.tex')

In [6]:
# ==============================================================================
# 5. TABLE 4 — GERMAN CREDIT CPU LATENCY (BASELINES + SCORE-NET)
# ==============================================================================
german_key = "german"
german_spec = DATASETS[german_key]
german_rows = df_profile[df_profile["Dataset"] == german_spec.display_name]
test_n = int(german_rows["Test_N"].iloc[0])

df_rows = []
path = baseline_path(german_key)
if path.exists():
    for _, row in pd.read_csv(path).iterrows():
        df_rows.append({
            "Model": row["Model"],
            "Category": row["Category"],
            "Acc": float(row["Accuracy"]) * 100.0,
            "Batch_ms": float(row["Inference_Time_Sec"]) * 1000.0,
            "ScoreNet": False,
        })
else:
    print(f"[WARNING] {path} is missing; only SCORE-Net rows will be written.")

for _, row in german_rows.iterrows():
    variant = "Global" if "Global" in row["Variant"] else "Compact"
    df_rows.append({
        "Model": f"SCORE-Net ({variant})",
        "Category": "Glass-Box",
        "Acc": float(row["Accuracy_%"]),
        "Batch_ms": float(row["CPU_Batch_ms"]),
        "ScoreNet": True,
    })

df_latency = pd.DataFrame(df_rows).sort_values("Acc", ascending=False)

# Single-query (batch = 1) latency is measured only for SCORE-Net. Baselines
# were timed as one batch over the full test set, so no single-query comparison
# is possible; a footnote below the table states this.
_single = german_rows.set_index("Variant")["CPU_Single_us"].astype(float)
scorenet_single_note = (
    f"{_single['Global Best (Peak Val Acc)']:.1f}~$\\mu$s for the Global Best "
    f"variant and {_single['Final Compact (Max Sparsity)']:.1f}~$\\mu$s for the "
    "Compact variant"
    if len(_single) else "about 65~$\\mu$s"
)

lines = [
    r"\begin{table*}[!t]",
    r"\centering",
    r"\caption{Host CPU inference latency on the German Credit test set "
    rf"($M_{{\text{{test}}}} = {test_n}$). The last column is batch latency "
    r"divided by $M_{\text{test}}$, an \emph{amortized} per-instance cost; "
    r"each model is timed through its own prediction API.}",
    r"\label{tab:cpu_latency_baselines}",
    r"\footnotesize",
    r"\renewcommand{\arraystretch}{0.85}",
    r"\setlength{\tabcolsep}{4pt}",
    r"\begin{tabular}{llccc}",
    r"\toprule",
    r"\textbf{Model} & \textbf{Category} & \textbf{Accuracy (\%)} & "
    r"\textbf{CPU Batch (ms)} & \textbf{Amortized per-Sample ($\mu$s)} \\",
    r"\midrule",
]
for _, row in df_latency.iterrows():
    per_sample = row["Batch_ms"] / test_n * 1000.0
    model = str(row["Model"]).replace("&", r"\&")
    if bool(row.get("ScoreNet", False)):
        model = rf"\textbf{{\modelname{{}} ({model.split('(')[1].rstrip(')')})}}"
    category = str(row["Category"]).replace("&", r"\&")
    lines.append(
        f"{model} & {category} & {row['Acc']:.2f} & {row['Batch_ms']:.2f} & "
        f"{per_sample:.1f} \\\\"
    )
lines += [
    r"\bottomrule",
    r"\end{tabular}",
        r"\par\smallskip",
    rf"\footnotesize\emph{{Note.}} One query costs {scorenet_single_note}; "
    r"baseline single queries were not measured. Baseline batch times are single "
    r"means, so differences below about 0.5~$\mu$s are within measurement noise. "
    r"SCORE-Net batch times are forward passes; baselines use their native "
    r"\texttt{predict} call.",
    r"\end{table*}",
]
write_table("tab_cpu_latency_baselines", lines, "CPU latency vs baselines")

% tab_cpu_latency_baselines — CPU latency vs baselines
\begin{table*}[!t]
\centering
\caption{Host CPU inference latency on the German Credit test set ($M_{\text{test}} = 100$). The last column is batch latency divided by $M_{\text{test}}$, an \emph{amortized} per-instance cost; each model is timed through its own prediction API.}
\label{tab:cpu_latency_baselines}
\footnotesize
\renewcommand{\arraystretch}{0.85}
\setlength{\tabcolsep}{4pt}
\begin{tabular}{llccc}
\toprule
\textbf{Model} & \textbf{Category} & \textbf{Accuracy (\%)} & \textbf{CPU Batch (ms)} & \textbf{Amortized per-Sample ($\mu$s)} \\
\midrule
Random Forest & SOTA Ensemble & 76.00 & 24.42 & 244.2 \\
\textbf{\modelname{} (Global)} & Glass-Box & 76.00 & 0.08 & 0.8 \\
\textbf{\modelname{} (Compact)} & Glass-Box & 75.00 & 0.08 & 0.8 \\
SVM (RBF) & Geometric \& DL & 73.00 & 2.46 & 24.6 \\
Logistic Regression & Interpretable & 73.00 & 0.14 & 1.4 \\
CatBoost & SOTA Ensemble & 73.00 & 0.35 & 3.5 \\
Decision Tree & Interpretable &

PosixPath('/home/ahmet/Projects/ScoreNetSingle/results/tables/tab_cpu_latency_baselines.tex')

In [7]:
# ==============================================================================
# 6. TABLE 5 — QUANTIZATION FIDELITY
# ==============================================================================
# Cell formatting follows two rules:
#
#   1. Half values round UP (ROUND_HALF_UP). Python's built-in format rounds
#      to even, so 96.625 prints as "96.62" and the row drifts 0.01 below the
#      observed value. Decimal consistently rounds half values up. "-0.00" is
#      normalized to "0.00".
#
#   2. Each row is internally consistent: Delta is the difference of the
#      printed Float and Scorecard values. Rounding the three columns
#      independently gave (85.30, 85.36, +0.07), which misses by 0.01 and
#      looks inconsistent to a reader checking the table.
def _dec(value):
    """Convert a value to a Decimal at 0.01 precision (half up)."""
    dec = Decimal(repr(float(value))).quantize(
        Decimal("0.01"), rounding=ROUND_HALF_UP
    )
    return abs(dec) if dec == 0 else dec


def _fmt(dec, signed=False):
    """Format a Decimal; no sign is written for zero."""
    if dec == 0:
        return f"{abs(dec):.2f}"
    return f"{dec:+.2f}" if signed else f"{dec:.2f}"


lines = [
    r"\begin{table*}[!tb]",
    r"\centering",
    r"\caption{Float versus rule-level integer scorecard. Only $W_f$ is quantized; $W_o$ stays "
    r"continuous. Fidelity is instance-level prediction agreement and $\Delta\text{Acc}$ is the "
    r"difference of the two displayed accuracies. The last block is the no-protected COMPAS "
    r"variant.}",
    r"\label{tab:scorecard_fidelity}",
    r"\renewcommand{\arraystretch}{1.3}",
    r"\setlength{\tabcolsep}{3pt}",
    r"\begin{tabular}{llccccccc}",
    r"\toprule",
    r"\textbf{Dataset} & \textbf{Variant} & \textbf{Rules ($H^*$)} & "
    r"\textbf{Float Acc (\%)} & \textbf{Scorecard Acc (\%)} & \textbf{$\Delta$ Acc (\%)} & "
    r"\textbf{Fidelity (\%)} & \textbf{Weight Range} & \textbf{Mean $\bar{\lambda}$} \\",
    r"\midrule",
]

for key in QUANT_KEYS:
    subset = df_quant[df_quant["Key"] == key]
    for index, (_, row) in enumerate(subset.iterrows()):
        label = QUANT_LABEL.get(key, row["Dataset"])
        dataset_cell = (
            rf"\multirow{{{len(subset)}}}{{*}}{{{label}}}" if index == 0 else ""
        )
        float_dec = _dec(row["Float_Acc_%"])
        score_dec = _dec(row["Scorecard_Acc_%"])
        lines.append(
            f"{dataset_cell} & {row['Variant']} & {int(row['H'])} & "
            f"{_fmt(float_dec)} & {_fmt(score_dec)} & "
            f"{_fmt(score_dec - float_dec, signed=True)} & {_fmt(_dec(row['Fidelity_%']))} & "
            f"$[{int(row['Integer_Min'])}, {int(row['Integer_Max'])}]$ & "
            f"{_fmt(_dec(row['Mean_Lambda']))} \\\\"
        )
    if key != QUANT_KEYS[-1]:
        lines.append(r"\midrule")

lines += [r"\bottomrule", r"\end{tabular}", r"\end{table*}"]
write_table("tab_scorecard_fidelity", lines, "quantization fidelity")

% tab_scorecard_fidelity — quantization fidelity
\begin{table*}[!tb]
\centering
\caption{Float versus rule-level integer scorecard. Only $W_f$ is quantized; $W_o$ stays continuous. Fidelity is instance-level prediction agreement and $\Delta\text{Acc}$ is the difference of the two displayed accuracies. The last block is the no-protected COMPAS variant.}
\label{tab:scorecard_fidelity}
\renewcommand{\arraystretch}{1.3}
\setlength{\tabcolsep}{3pt}
\begin{tabular}{llccccccc}
\toprule
\textbf{Dataset} & \textbf{Variant} & \textbf{Rules ($H^*$)} & \textbf{Float Acc (\%)} & \textbf{Scorecard Acc (\%)} & \textbf{$\Delta$ Acc (\%)} & \textbf{Fidelity (\%)} & \textbf{Weight Range} & \textbf{Mean $\bar{\lambda}$} \\
\midrule
\multirow{2}{*}{COMPAS} & Global Best & 11 & 68.33 & 68.40 & +0.07 & 99.79 & $[-3, 3]$ & 6.27 \\
 & Compact & 5 & 66.32 & 66.32 & 0.00 & 100.00 & $[-3, 3]$ & 7.60 \\
\midrule
\multirow{2}{*}{German Credit} & Global Best & 64 & 76.00 & 77.00 & +1.00 & 97.00 & $[-9, 8]$ & 29.11 

PosixPath('/home/ahmet/Projects/ScoreNetSingle/results/tables/tab_scorecard_fidelity.tex')

In [8]:
# ==============================================================================
# 7. TABLE 6 — SYNTHETIC ABLATION (CACHE FROM NOTEBOOK 4)
# ==============================================================================
ablation_path = require_artifact(
    RESULTS_DIR / "ablation_synthetic.csv",
    "notebooks/06_ablations.ipynb",
)
df_ablation = pd.read_csv(ablation_path)
reference = df_ablation[df_ablation["Is Reference"]].iloc[0]

reg = df_ablation[df_ablation["Group"] == "regularization"]
cap = df_ablation[(df_ablation["Group"] == "capacity") | df_ablation["Is Reference"]]
cap = cap.sort_values("Initial H")

lines = [
    r"\begin{table*}[t]",
    r"\centering",
    r"\caption{Ablation study on the multi-class Synthetic benchmark "
    rf"({int(splits['synthetic'].n_classes)} classes, "
    rf"{int(splits['synthetic'].n_features)} features). We vary individual loss "
    r"regularization components and the initial model capacity $H$ and report test "
    r"accuracy, rule pruning, feature sparsity and average rule length.}",
    r"\label{tab:ablation_synthetic}",
    r"\setlength{\tabcolsep}{3pt}",
    r"\begin{tabular}{l ccc ccc c}",
    r"\toprule",
    r"\textbf{Ablation Variant} & \textbf{Initial $H$} & \textbf{Pruned $H^*$} & "
    r"\textbf{Pruning (\%)} & \textbf{Feat. Sparsity (\%)} & \textbf{Avg. Feat/Rule} & "
    r"\textbf{Test Acc. (\%)} & \textbf{$\Delta$ Acc. (\%)} \\",
    r"\midrule",
    rf"\multicolumn{{8}}{{l}}{{\textit{{\textbf{{Loss Regularization and Dropout Ablation "
    rf"($H_{{\text{{init}}}}={int(reference['Initial H'])}$)}}}}}} \\",
]


def ablation_row(row, label, bold_rules=False):
    """Convert one ablation row to LaTeX; ``label`` changes per section."""
    delta = (
        r"\textit{Baseline}" if bool(row["Is Reference"])
        else f"{row['Delta Acc (%)']:+.2f}\\%"
    )
    rules = int(row["Pruned Rules (H*)"])
    rules_tex = rf"\textbf{{{rules}}}" if bold_rules else str(rules)
    acc_tex = f"{row['Test Acc (%)']:.2f}\\%"
    if bool(row["Is Reference"]):
        acc_tex = rf"\textbf{{{acc_tex}}}"
    return (
        f"{label} & {int(row['Initial H'])} & {rules_tex} & "
        f"{row['Rule Pruning (%)']:.1f}\\% & {row['Feat Sparsity (%)']:.1f}\\% & "
        f"{row['Avg Feats/Rule']:.1f} & {acc_tex} & {delta} \\\\"
    )


best_reg_rules = int(reg["Pruned Rules (H*)"].max())
for _, row in reg.iterrows():
    lines.append(
        ablation_row(
            row,
            row["Variant"],
            bold_rules=int(row["Pruned Rules (H*)"]) == best_reg_rules,
        )
    )

lines += [r"\midrule",
          r"\multicolumn{8}{l}{\textit{\textbf{Initial Capacity Scaling ($H$)}}} \\"]
best_cap_rules = int(cap["Pruned Rules (H*)"].max())
reference_H = int(reference["Initial H"])
for _, row in cap.iterrows():
    # The reference row is labeled "Standard Capacity" in this section.
    label = (
        f"Standard Capacity ($H={reference_H}$)"
        if bool(row["Is Reference"])
        else row["Capacity TeX Label"]
    )
    lines.append(
        ablation_row(
            row, label, bold_rules=int(row["Pruned Rules (H*)"]) == best_cap_rules
        )
    )

lines += [r"\bottomrule", r"\end{tabular}", r"\end{table*}"]
write_table("tab_ablation_synthetic", lines, "synthetic ablation")

% tab_ablation_synthetic — synthetic ablation
\begin{table*}[t]
\centering
\caption{Ablation study on the multi-class Synthetic benchmark (20 classes, 30 features). We vary individual loss regularization components and the initial model capacity $H$ and report test accuracy, rule pruning, feature sparsity and average rule length.}
\label{tab:ablation_synthetic}
\setlength{\tabcolsep}{3pt}
\begin{tabular}{l ccc ccc c}
\toprule
\textbf{Ablation Variant} & \textbf{Initial $H$} & \textbf{Pruned $H^*$} & \textbf{Pruning (\%)} & \textbf{Feat. Sparsity (\%)} & \textbf{Avg. Feat/Rule} & \textbf{Test Acc. (\%)} & \textbf{$\Delta$ Acc. (\%)} \\
\midrule
\multicolumn{8}{l}{\textit{\textbf{Loss Regularization and Dropout Ablation ($H_{\text{init}}=1024$)}}} \\
SCORE-Net (Full Dual $L_1$, H=1024) & 1024 & 565 & 44.8\% & 78.0\% & 6.6 & \textbf{90.03\%} & \textit{Baseline} \\
No Dropout ($p = 0.0$) & 1024 & 437 & 57.3\% & 71.5\% & 8.5 & 91.22\% & +1.19\% \\
No Feature $L_1$ ($\lambda_1 = 0$) & 1024 &

PosixPath('/home/ahmet/Projects/ScoreNetSingle/results/tables/tab_ablation_synthetic.tex')

In [9]:
# ==============================================================================
# 8. TABLE 7 — PROPOSITION 1 VALIDATION (EXACT BIAS SHIFTING FOR PRUNED RULES)
# ==============================================================================
theorem_path = require_artifact(
    RESULTS_DIR / "theorem1_bias_shifting.csv",
    "notebooks/06_ablations.ipynb",
)
df_theorem = pd.read_csv(theorem_path)

max_drift = float(df_theorem["Thm1 Max Drift"].max())
min_fidelity = float(df_theorem["Thm1 Fidelity (%)"].min())
max_penalty = float(df_theorem["Drift Penalty (%)"].max())


def sci(value):
    """Scientific notation; dollar signs are added by the caller."""
    mantissa, exponent = f"{value:.2e}".split("e")
    return rf"{mantissa} \times 10^{{{int(exponent)}}}"


penalty_sentence = (
    rf"Naive pruning induces up to a {max_penalty:.2f}\% test-accuracy penalty."
    if max_penalty > 0.005
    else "On these checkpoints the naive accuracy penalty is negligible."
)

lines = [
    r"\begin{table*}[t]",
    r"\centering",
    r"\caption{Exact bias shifting versus naive removal of structurally empty rules, so the only "
    rf"difference is the bias update. Proposition~\ref{{prop:bias_shifting}} preserves the output up "
    rf"to floating-point and threshold effects (maximum drift ${sci(max_drift)}$).}}",
    r"\label{tab:theorem1_bias_shifting}",
    r"\resizebox{\textwidth}{!}{%",
    r"\begin{tabular}{l c cc cc c}",
    r"\toprule",
    r"\multirow{2}{*}{\textbf{Dataset}} & \multirow{2}{*}{\textbf{Pruned Rules}} & "
    r"\multicolumn{2}{c}{\textbf{Proposition~\ref{prop:bias_shifting} (Exact Shift)}} & "
    r"\multicolumn{2}{c}{\textbf{Naive Pruning (No Shift)}} & "
    r"\multirow{2}{*}{\textbf{Drift Penalty}} \\",
    r"\cmidrule(lr){3-4} \cmidrule(lr){5-6}",
    r" & & \textbf{Max Drift} & \textbf{Fidelity} & \textbf{Max Drift} & "
    r"\textbf{Fidelity} & \\",
    r"\midrule",
]

for _, row in df_theorem.iterrows():
    lines.append(
        f"{row['Dataset']} & {int(row['Eliminated Rules'])} / {int(row['Initial H'])} & "
        f"${sci(row['Thm1 Max Drift'])}$ & {row['Thm1 Fidelity (%)']:.2f}\\% & "
        f"{row['Naive Max Drift']:.4f} & {row['Naive Fidelity (%)']:.2f}\\% & "
        f"{row['Drift Penalty (%)']:+.2f}\\% \\\\"
    )

lines += [r"\bottomrule", r"\end{tabular}%", r"}", r"\end{table*}"]
write_table("tab_theorem1_bias_shifting", lines, "Proposition 1 validation")

% tab_theorem1_bias_shifting — Proposition 1 validation
\begin{table*}[t]
\centering
\caption{Exact bias shifting versus naive removal of structurally empty rules, so the only difference is the bias update. Proposition~\ref{prop:bias_shifting} preserves the output up to floating-point and threshold effects (maximum drift $5.72 \times 10^{-6}$).}
\label{tab:theorem1_bias_shifting}
\resizebox{\textwidth}{!}{%
\begin{tabular}{l c cc cc c}
\toprule
\multirow{2}{*}{\textbf{Dataset}} & \multirow{2}{*}{\textbf{Pruned Rules}} & \multicolumn{2}{c}{\textbf{Proposition~\ref{prop:bias_shifting} (Exact Shift)}} & \multicolumn{2}{c}{\textbf{Naive Pruning (No Shift)}} & \multirow{2}{*}{\textbf{Drift Penalty}} \\
\cmidrule(lr){3-4} \cmidrule(lr){5-6}
 & & \textbf{Max Drift} & \textbf{Fidelity} & \textbf{Max Drift} & \textbf{Fidelity} & \\
\midrule
COMPAS & 27 / 32 & $1.19 \times 10^{-7}$ & 100.00\% & 0.3647 & 62.44\% & +9.01\% \\
COMPAS (no protected attributes) & 29 / 32 & $8.94 \times 10^{-8}$ & 100

PosixPath('/home/ahmet/Projects/ScoreNetSingle/results/tables/tab_theorem1_bias_shifting.tex')

In [10]:
# ==============================================================================
# 10. STATISTICAL RELIABILITY (FOR SMALL TEST SETS)
# ==============================================================================
# COMPAS (1,443) and especially German Credit (100) have small test sets:
# on German Credit one sample is 1.0 percentage point. A single accuracy
# percentage hides sampling noise, so four measures are reported:
#
#   1. Stratified bootstrap 95% confidence interval (SCORE-Net accuracy),
#   2. McNemar test (SCORE-Net vs. best baseline, paired),
#   3. Paired bootstrap difference CI,
#   4. Minimum detectable difference (depends on test size).
from utils.statistics import (
    bootstrap_ci,
    mcnemar_test,
    minimum_detectable_difference,
    paired_bootstrap_diff,
)

PRED_DIR = RESULTS_DIR / "predictions"
stats_rows = []
missing_predictions = []

for key in KEYS:
    spec = DATASETS[key]
    split = splits[key]
    train_loader = split.loader("train")
    y_true = split.y_test.cpu().numpy()

    # Baseline predictions (if available)
    base_preds, base_names = {}, []
    pred_path = PRED_DIR / f"baselines_{key}_predictions.npz"
    if pred_path.exists():
        archive = np.load(pred_path)
        for name in archive.files:
            if name == "y_true":
                continue
            base_preds[name] = archive[name]
            base_names.append(name)
    else:
        missing_predictions.append(key)

    # Baseline accuracies (from CSV) → choose the best baseline
    bench_path = baseline_path(key)
    best_baseline_name, best_baseline_acc = None, -1.0
    if bench_path.exists():
        df_b = pd.read_csv(bench_path)
        for _, row in df_b.iterrows():
            name = row["Model"]
            if name in base_preds:
                acc = float(np.mean(base_preds[name] == y_true)) * 100.0
                if acc > best_baseline_acc:
                    best_baseline_acc, best_baseline_name = acc, name

    for variant, checkpoint in [
        ("Global Best", spec.global_checkpoint),
        ("Compact", spec.final_checkpoint),
    ]:
        model = compact_model(load_checkpoint(checkpoint), train_loader)
        with torch.no_grad():
            probs = torch.softmax(model(split.X_test), dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)

        acc_ci = bootstrap_ci(y_true, preds, metric="accuracy")
        f1_ci = bootstrap_ci(y_true, preds, metric="macro_f1")

        # Validation → test gap (sensitivity of model selection to noise)
        with torch.no_grad():
            val_preds = model(split.X_val).argmax(dim=1).cpu().numpy()
        val_acc = float(np.mean(val_preds == split.y_val.cpu().numpy())) * 100.0
        val_ci = bootstrap_ci(
            split.y_val.cpu().numpy(), val_preds, metric="accuracy"
        )

        row = {
            "Dataset": spec.display_name,
            "Key": key,
            "Variant": variant,
            "H": int(model.H),
            "Test_N": int(y_true.size),
            "Val_N": int(split.y_val.shape[0]),
            "Acc_%": acc_ci.point * 100.0,
            "Acc_CI_Low": acc_ci.ci_low * 100.0,
            "Acc_CI_High": acc_ci.ci_high * 100.0,
            "F1_%": f1_ci.point * 100.0,
            "F1_CI_Low": f1_ci.ci_low * 100.0,
            "F1_CI_High": f1_ci.ci_high * 100.0,
            "Val_Acc_%": val_acc,
            "Val_CI_Low": val_ci.ci_low * 100.0,
            "Val_CI_High": val_ci.ci_high * 100.0,
            "Val_Test_Gap": val_acc - acc_ci.point * 100.0,
            "Min_Detectable_Diff_pp": minimum_detectable_difference(int(y_true.size)),
        }

        if best_baseline_name is not None:
            base_pred = base_preds[best_baseline_name]
            mcnemar = mcnemar_test(y_true, preds, base_pred)
            diff = paired_bootstrap_diff(y_true, preds, base_pred)
            row.update({
                "Best_Baseline": best_baseline_name,
                "Best_Baseline_Acc_%": best_baseline_acc,
                "Diff_vs_Baseline_pp": acc_ci.point * 100.0 - best_baseline_acc,
                "Diff_CI_Low_pp": diff.diff_ci_low * 100.0,
                "Diff_CI_High_pp": diff.diff_ci_high * 100.0,
                "McNemar_p": mcnemar.p_value,
                "McNemar_Test": mcnemar.test,
                "Only_B_Correct": mcnemar.a_wrong_b_right,
                "Only_A_Correct": mcnemar.a_right_b_wrong,
            })

        stats_rows.append(row)

df_stats = pd.DataFrame(stats_rows)
stats_path = RESULTS_DIR / "statistical_tests.csv"
df_stats.to_csv(stats_path, index=False)
print(f"Saved: {stats_path}")
if missing_predictions:
    print(
        "\n[WARNING] No baseline prediction cache for these datasets; "
        "McNemar test skipped: " + ", ".join(missing_predictions)
        + "\n         Re-run the corresponding baseline notebook to fix this."
    )

pd.set_option("display.width", 250)
print(
    df_stats[[
        "Dataset", "Variant", "Test_N", "Acc_%", "Acc_CI_Low", "Acc_CI_High",
        "Val_Acc_%", "Val_Test_Gap", "Best_Baseline", "Best_Baseline_Acc_%",
        "McNemar_p",
    ]].to_string(index=False)
)

# ------------------------------------------------------------------------------
# LaTeX table
# ------------------------------------------------------------------------------
def _ci_cell(point, low, high, decimals=2):
    return f"{point:.{decimals}f} [{low:.{decimals}f}, {high:.{decimals}f}]"


def _p_cell(p_value):
    if pd.isna(p_value):
        return "--"
    if p_value < 0.001:
        return r"$<0.001$"
    return f"{p_value:.3f}"


german_test_tex = f"{splits['german'].n_test:,}".replace(",", "{,}")
adult_test_tex = f"{splits['adult'].n_test:,}".replace(",", "{,}")

lines = [
    r"\begin{table*}[t]",
    r"\centering",
    rf"\caption{{Stratified-bootstrap $95\%$ confidence intervals, validation--test gap and paired "
    r"McNemar test against the strongest baseline (listed beneath the table). Test sets range from "
    rf"$M_{{\text{{test}}}}={german_test_tex}$ to $M_{{\text{{test}}}}={adult_test_tex}$, so a bare "
    r"accuracy is not comparable across datasets. $p>0.05$ means the difference is not "
    r"distinguishable from sampling noise.}",
    r"\label{tab:statistical_reliability}",
    r"\setlength{\tabcolsep}{4pt}",
    r"\begin{tabular}{ll c cc cc c c}",
    r"\toprule",
    r"\textbf{Dataset} & \textbf{Variant} & \textbf{Test $M$} & "
    r"\textbf{Acc (\%) [95\% CI]} & \textbf{Macro-F1 [95\% CI]} & "
    r"\textbf{Val Acc [95\% CI]} & \textbf{Val$-$Test} & \textbf{$\Delta$ (pp)} & "
    r"\textbf{McNemar $p$} \\",
    r"\midrule",
]

for key in KEYS:
    sub = df_stats[df_stats["Key"] == key]
    for index, (_, row) in enumerate(sub.iterrows()):
        dataset_cell = (
            rf"\multirow{{{len(sub)}}}{{*}}{{{row['Dataset']}}}" if index == 0 else ""
        )
        if "Best_Baseline" in row and not pd.isna(row.get("McNemar_p")):
            delta_cell = f"{row['Diff_vs_Baseline_pp']:+.2f}"
            p_cell = _p_cell(row["McNemar_p"])
        else:
            delta_cell, p_cell = "--", "--"
        lines.append(
            f"{dataset_cell} & {row['Variant']} & {int(row['Test_N']):,} & "
            f"{_ci_cell(row['Acc_%'], row['Acc_CI_Low'], row['Acc_CI_High'])} & "
            f"{_ci_cell(row['F1_%'], row['F1_CI_Low'], row['F1_CI_High'])} & "
            f"{_ci_cell(row['Val_Acc_%'], row['Val_CI_Low'], row['Val_CI_High'])} & "
            f"{row['Val_Test_Gap']:+.2f} & {delta_cell} & {p_cell} \\\\"
        )
    if key != KEYS[-1]:
        lines.append(r"\midrule")

# The best baseline is the same for both variants of a dataset; to keep the
# table body narrow, baseline names are listed on one line below the table.
baseline_notes = []
for key in KEYS:
    sub = df_stats[(df_stats["Key"] == key) & df_stats["Best_Baseline"].notna()]
    if not sub.empty:
        best = sub.iloc[0]
        baseline_notes.append(
            f"\\textbf{{{best['Dataset']}}}: {best['Best_Baseline']} "
            f"({best['Best_Baseline_Acc_%']:.2f}\\%)"
        )

lines += [r"\bottomrule", r"\end{tabular}"]
if baseline_notes:
    lines += [
        r"\par\vspace{1pt}",
        r"{\footnotesize Best baseline per dataset (highest observed test "
        r"accuracy): " + "; ".join(baseline_notes) + r".}",
    ]
lines += [r"\end{table*}"]

write_table("tab_statistical_reliability", lines, "sampling uncertainty and paired tests")

Saved: /home/ahmet/Projects/ScoreNetSingle/results/statistical_tests.csv
      Dataset     Variant  Test_N     Acc_%  Acc_CI_Low  Acc_CI_High  Val_Acc_%  Val_Test_Gap       Best_Baseline  Best_Baseline_Acc_%     McNemar_p
       COMPAS Global Best    1443 68.329868   65.904366    70.755371  70.588235      2.258367 Logistic Regression            68.745669  7.139041e-01
       COMPAS     Compact    1443 66.320166   63.894664    68.745669  69.723183      3.403017 Logistic Regression            68.745669  1.811968e-02
German Credit Global Best     100 76.000000   68.000000    83.000000  79.000000      3.000000       Random Forest            76.000000  1.000000e+00
German Credit     Compact     100 75.000000   67.000000    82.025000  76.000000      1.000000       Random Forest            76.000000  1.000000e+00
    UCI Adult Global Best    9045 85.295744   84.632117    85.981205  86.180210      0.884467   Gradient Boosting            86.865672  3.319361e-12
    UCI Adult     Compact    9045

PosixPath('/home/ahmet/Projects/ScoreNetSingle/results/tables/tab_statistical_reliability.tex')

In [11]:
# ==============================================================================
# 11. ALL DATASETS — 5x5 REPEATED STRATIFIED CROSS-VALIDATION
# ==============================================================================
# Every model is evaluated on the same 25 folds: the outer test fold is 20% of
# the dataset and validation is 20% of the remaining pool. The table is built
# entirely from results/cv_<key>.csv and results/cv_baselines_<key>.csv; no
# number is hard-coded here.
from utils.experiments import CV_FOLDS, CV_REPEATS, cv_results_path

CV_KEYS = ["compas", "german", "adult", "synthetic"]
VARIANTS = [("test_global", "Global Best"), ("test_final", "Compact")]

cv_rows = []
n_folds_seen = set()

for key in CV_KEYS:
    spec = DATASETS[key]
    sn = pd.read_csv(require_artifact(
        cv_results_path(key),
        f"python scripts/run_cv.py --models scorenet --datasets {key}",
    ))
    bl = pd.read_csv(require_artifact(
        RESULTS_DIR / f"cv_baselines_{key}.csv",
        f"python scripts/run_cv.py --models baselines --datasets {key}",
    ))

    # Reference: the baseline with the highest mean CV accuracy.
    best_model = bl.groupby("model")["test_accuracy"].mean().idxmax()
    bm = bl[bl["model"] == best_model]

    # Only folds completed on both sides are paired.
    pair = sn[["repeat", "fold"] + [c for c, _ in VARIANTS]].merge(
        bm[["repeat", "fold", "test_accuracy"]], on=["repeat", "fold"], how="inner"
    )
    if pair.empty:
        raise RuntimeError(f"{key}: No common fold found for SCORE-Net and the baseline.")
    n_folds_seen.add(len(pair))

    base_mean = bm["test_accuracy"].mean()
    base_std = bm["test_accuracy"].std(ddof=1)

    for column, variant in VARIANTS:
        acc = pair[column].to_numpy() * 100.0
        delta = acc - pair["test_accuracy"].to_numpy()
        cv_rows.append({
            "dataset": spec.display_name,
            "variant": variant,
            "acc": f"{acc.mean():.2f}$\\pm${acc.std(ddof=1):.2f}",
            "baseline": best_model,
            "base": f"{base_mean:.2f}$\\pm${base_std:.2f}",
            "delta": delta.mean(),
            "won": int((delta > 0).sum()),
            "folds": len(delta),
        })

if len(n_folds_seen) != 1:
    raise RuntimeError(f"Fold counts differ across datasets: {sorted(n_folds_seen)}")
n_folds = n_folds_seen.pop()
if n_folds != CV_REPEATS * CV_FOLDS:
    raise RuntimeError(
        f"Beklenen {CV_REPEATS * CV_FOLDS} kat, bulunan {n_folds}. "
        "run_cv.py may be incomplete."
    )


def _delta_cell(value):
    """Signed difference; bold when SCORE-Net is ahead."""
    text = f"{value:+.2f}"
    return rf"\textbf{{{text}}}" if value > 0 else text


lines = [
    r"\begin{table*}[t]",
    r"\centering",
    rf"\caption{{Repeated stratified ${CV_REPEATS}\times{CV_FOLDS}$ cross-validation. All models "
    rf"share the same ${n_folds}$ folds, with a 20\% outer test fold and a 20\% inner validation "
    r"fold, so the test fold is never used for model selection. Entries are mean $\pm$ standard "
    r"deviation; $\Delta$ is the paired per-fold difference to the strongest baseline and the last "
    r"column counts folds won.}",
    r"\label{tab:cross_validation}",
    r"\setlength{\tabcolsep}{4pt}",
    r"\begin{tabular}{ll c ll c c}",
    r"\toprule",
    r"\textbf{Dataset} & \textbf{Variant} & \textbf{CV Acc (\%)} & "
    r"\textbf{Best baseline} & \textbf{Baseline CV (\%)} & "
    r"\textbf{$\Delta$ (pp)} & \textbf{Folds won} \\",
    r"\midrule",
]

for dataset in ORDER:
    block = [row for row in cv_rows if row["dataset"] == dataset]
    if not block:
        continue
    for index, row in enumerate(block):
        dataset_cell = (
            rf"\multirow{{{len(block)}}}{{*}}{{{dataset}}}" if index == 0 else ""
        )
        baseline_cell = (
            rf"\multirow{{{len(block)}}}{{*}}{{{row['baseline']}}}" if index == 0 else ""
        )
        base_cell = (
            rf"\multirow{{{len(block)}}}{{*}}{{{row['base']}}}" if index == 0 else ""
        )
        lines.append(
            f"{dataset_cell} & {row['variant']} & {row['acc']} & {baseline_cell} & "
            f"{base_cell} & {_delta_cell(row['delta'])} & "
            f"{row['won']}/{row['folds']} \\\\"
        )
    if dataset != ORDER[-1]:
        lines.append(r"\midrule")

lines += [r"\bottomrule", r"\end{tabular}", r"\end{table*}"]
write_table("tab_cross_validation", lines, "5x5 CV, all datasets")


% tab_cross_validation — 5x5 CV, all datasets
\begin{table*}[t]
\centering
\caption{Repeated stratified $5\times5$ cross-validation. All models share the same $25$ folds, with a 20\% outer test fold and a 20\% inner validation fold, so the test fold is never used for model selection. Entries are mean $\pm$ standard deviation; $\Delta$ is the paired per-fold difference to the strongest baseline and the last column counts folds won.}
\label{tab:cross_validation}
\setlength{\tabcolsep}{4pt}
\begin{tabular}{ll c ll c c}
\toprule
\textbf{Dataset} & \textbf{Variant} & \textbf{CV Acc (\%)} & \textbf{Best baseline} & \textbf{Baseline CV (\%)} & \textbf{$\Delta$ (pp)} & \textbf{Folds won} \\
\midrule
\multirow{2}{*}{COMPAS} & Global Best & 67.56$\pm$1.49 & \multirow{2}{*}{CatBoost} & \multirow{2}{*}{67.94$\pm$1.17} & -0.38 & 9/25 \\
 & Compact & 66.81$\pm$1.66 &  &  & -1.13 & 5/25 \\
\midrule
\multirow{2}{*}{German Credit} & Global Best & 72.72$\pm$2.78 & \multirow{2}{*}{CatBoost} & \multirow{2

PosixPath('/home/ahmet/Projects/ScoreNetSingle/results/tables/tab_cross_validation.tex')

## Generated tables

| File | Table |
| --- | --- |
| `results/tables/tab_dataset_summary.tex` | Benchmark dataset summary |
| `results/tables/tab_benchmark_comparison.tex` | Comprehensive benchmark comparison |
| `results/tables/tab_computational_efficiency.tex` | Resource profile and CPU/GPU latency |
| `results/tables/tab_cpu_latency_baselines.tex` | German Credit CPU latency |
| `results/tables/tab_scorecard_fidelity.tex` | Quantization fidelity |
| `results/tables/tab_ablation_synthetic.tex` | Synthetic ablation |
| `results/tables/tab_theorem1_bias_shifting.tex` | Proposition 1 validation |
| `results/tables/tab_statistical_reliability.tex` | Bootstrap CIs and McNemar tests |
| `results/tables/tab_cross_validation.tex` | 5×5 CV, all datasets |

### Verification notes

* All numbers come from checkpoints, `results/` caches, or data splits. The
  notebook contains no hard-coded accuracy, rule count, or latency constant.
* Quantization applies only to `W_f`; `W_o` stays continuous. The budget
  (Λ_max, τ) is defined per dataset and variant in `utils.experiments`; the
  table, figure, and analysis notebooks read the same source.
* In the benchmark table, the best score within a family is **bold** and the
  best overall per dataset is **underlined**. Ties may mark several cells.
* If baseline CSVs are missing, only SCORE-Net rows are written for that
  dataset and the notebook prints a warning; no numbers are invented.


In [12]:
# ==============================================================================
# 12. TRAINING COST — SCORE-Net vs. baselines
# ==============================================================================
# SCORE-Net timing is the full pruning trajectory, averaged over
# cross-validation folds (results/cv_<key>.csv, elapsed_s). Baseline training
# times on the same folds are read from results/cv_baselines_<key>.csv.
# SCORE-Net times are taken on the device chosen by the CV runner (GPU when
# CUDA is available); baselines train on CPU. The table is therefore an
# order-of-magnitude comparison, not a controlled hardware benchmark.
from utils.experiments import cv_results_path

COST_KEYS = ["compas", "german", "adult", "synthetic"]

cost_rows = []
for key in COST_KEYS:
    spec = DATASETS[key]
    cv = pd.read_csv(require_artifact(
        cv_results_path(key),
        f"python scripts/run_cv.py --models scorenet --datasets {key}",
    ))
    bl = pd.read_csv(require_artifact(
        RESULTS_DIR / f"cv_baselines_{key}.csv",
        f"python scripts/run_cv.py --models baselines --datasets {key}",
    ))
    seconds = cv["elapsed_s"].to_numpy()
    stages = int(spec.prune_iters) + 1
    cost_rows.append({
        "dataset": spec.display_name,
        "n_train": int(cv["n_train"].iloc[0]),
        "sn_mean": seconds.mean(),
        "sn_std": seconds.std(ddof=1),
        "sn_stage": seconds.mean() / stages,
        "bl_median": float(bl["elapsed_s"].median()),
        "bl_fastest": float(bl["elapsed_s"].min()),
    })


def _fmt_minutes(seconds):
    minutes = seconds / 60.0
    return f"{minutes:.1f}" if minutes >= 1 else f"{minutes:.2f}"


def _fmt_seconds(seconds):
    return "$<$0.01" if seconds < 0.005 else f"{seconds:.2f}"


lines = [
    r"\begin{table}[!b]",
    r"\centering",
    r"\caption{Offline training cost. \modelname{} is the mean $\pm$ standard deviation of the "
    r"full pruning trajectory (base fit plus 14--20 stages); per stage divides this by the number of "
    r"stages. Baselines are the median and minimum over the 13 models. The two are not a "
    r"like-for-like speed ratio.}",
    r"\label{tab:train_cost}",
    r"\setlength{\tabcolsep}{4pt}",
    r"\begin{tabular}{l r c c c c}",
    r"\toprule",
    r"\textbf{Dataset} & \textbf{Fold $M_{\text{train}}$} & "
    r"\textbf{\modelname{} trajectory (min)} & \textbf{Per stage (s)} & "
    r"\textbf{Baseline median} & \textbf{Baseline min} \\",
    r"\midrule",
]

for row in cost_rows:
    lines.append(
        f"{row['dataset']} & {row['n_train']:,} & "
        f"{_fmt_minutes(row['sn_mean'])}$\\pm${_fmt_minutes(row['sn_std'])} & "
        f"{row['sn_stage']:.2f} & "
        f"{_fmt_seconds(row['bl_median'])}" + "\\,s & " +
        f"{_fmt_seconds(row['bl_fastest'])}" + "\\,s \\\\"
    )

lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
write_table("tab_train_cost", lines, "offline training cost")


% tab_train_cost — offline training cost
\begin{table}[!b]
\centering
\caption{Offline training cost. \modelname{} is the mean $\pm$ standard deviation of the full pruning trajectory (base fit plus 14--20 stages); per stage divides this by the number of stages. Baselines are the median and minimum over the 13 models. The two are not a like-for-like speed ratio.}
\label{tab:train_cost}
\setlength{\tabcolsep}{4pt}
\begin{tabular}{l r c c c c}
\toprule
\textbf{Dataset} & \textbf{Fold $M_{\text{train}}$} & \textbf{\modelname{} trajectory (min)} & \textbf{Per stage (s)} & \textbf{Baseline median} & \textbf{Baseline min} \\
\midrule
COMPAS & 4,616 & 2.9$\pm$0.60 & 8.80 & 0.11\,s & 0.01\,s \\
German Credit & 640 & 0.23$\pm$0.03 & 0.93 & 0.03\,s & $<$0.01\,s \\
UCI Adult & 28,941 & 5.2$\pm$0.58 & 19.65 & 2.80\,s & 0.02\,s \\
Synthetic & 12,800 & 7.9$\pm$0.91 & 22.63 & 27.74\,s & 0.01\,s \\
\bottomrule
\end{tabular}
\end{table}



PosixPath('/home/ahmet/Projects/ScoreNetSingle/results/tables/tab_train_cost.tex')

In [13]:
# ==============================================================================
# 13. PROTECTED-ATTRIBUTE ABLATION — COMPAS, sex/race removed
# ==============================================================================
# Same dataset, split ratios, and hyperparameters; the only change is dropping
# the one-hot sex_* and race_* columns from the input.
from utils.experiments import cv_results_path

ablation_rows = []
for key in ["compas", "compas_noprot"]:
    spec = DATASETS[key]
    cv = pd.read_csv(require_artifact(
        cv_results_path(key),
        f"python scripts/run_cv.py --models all --datasets {key}",
    ))
    bl = pd.read_csv(require_artifact(
        RESULTS_DIR / f"cv_baselines_{key}.csv",
        f"python scripts/run_cv.py --models baselines --datasets {key}",
    ))
    best_model = bl.groupby("model")["test_accuracy"].mean().idxmax()
    pair = cv[["repeat", "fold", "test_global", "test_final"]].merge(
        bl[bl["model"] == best_model][["repeat", "fold", "test_accuracy"]],
        on=["repeat", "fold"], how="inner",
    )
    for column, variant in [("test_global", "Global Best"), ("test_final", "Compact")]:
        acc = pair[column].to_numpy() * 100.0
        delta = acc - pair["test_accuracy"].to_numpy()
        ablation_rows.append({
            "label": "All features" if key == "compas" else "No sex/race",
            "variant": variant,
            "n_features": 13 if key == "compas" else 7,
            "acc": f"{acc.mean():.2f}$\\pm${acc.std(ddof=1):.2f}",
            "best": best_model,
            "delta": delta.mean(),
            "won": int((delta > 0).sum()),
        })

lines = [
    r"\begin{table}[t]",
    r"\centering",
    r"\caption{Removing the legally protected attributes from COMPAS. Split ratios and "
    r"hyperparameters are unchanged; only \texttt{sex\_*} and \texttt{race\_*} are dropped, "
    r"leaving 7 instead of 13 inputs. Values are $5\times5$ cross-validated test accuracy and "
    r"$\Delta$ is the paired per-fold difference to the strongest baseline.}",
    r"\label{tab:protected_ablation}",
    r"\setlength{\tabcolsep}{4pt}",
    r"\begin{tabular}{l l r c l c c}",
    r"\toprule",
    r"\textbf{Feature set} & \textbf{Variant} & \textbf{$N$} & "
    r"\textbf{CV Acc (\%)} & \textbf{Best baseline} & \textbf{$\Delta$ (pp)} & "
    r"\textbf{Folds} \\",
    r"\midrule",
]
for row in ablation_rows:
    delta_cell = f"{row['delta']:+.2f}"
    if row["delta"] > 0:
        delta_cell = rf"\textbf{{{delta_cell}}}"
    lines.append(
        f"{row['label']} & {row['variant']} & {row['n_features']} & {row['acc']} & "
        f"{row['best']} & {delta_cell} & {row['won']}/25 \\\\"
    )
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
write_table("tab_protected_ablation", lines, "protected-attribute ablation")


% tab_protected_ablation — protected-attribute ablation
\begin{table}[t]
\centering
\caption{Removing the legally protected attributes from COMPAS. Split ratios and hyperparameters are unchanged; only \texttt{sex\_*} and \texttt{race\_*} are dropped, leaving 7 instead of 13 inputs. Values are $5\times5$ cross-validated test accuracy and $\Delta$ is the paired per-fold difference to the strongest baseline.}
\label{tab:protected_ablation}
\setlength{\tabcolsep}{4pt}
\begin{tabular}{l l r c l c c}
\toprule
\textbf{Feature set} & \textbf{Variant} & \textbf{$N$} & \textbf{CV Acc (\%)} & \textbf{Best baseline} & \textbf{$\Delta$ (pp)} & \textbf{Folds} \\
\midrule
All features & Global Best & 13 & 67.56$\pm$1.49 & CatBoost & -0.38 & 9/25 \\
All features & Compact & 13 & 66.81$\pm$1.66 & CatBoost & -1.13 & 5/25 \\
No sex/race & Global Best & 7 & 67.67$\pm$1.35 & EBM (InterpretML) & -0.43 & 3/25 \\
No sex/race & Compact & 7 & 65.62$\pm$2.42 & EBM (InterpretML) & -2.47 & 2/25 \\
\bottomrule
\end

PosixPath('/home/ahmet/Projects/ScoreNetSingle/results/tables/tab_protected_ablation.tex')

In [14]:
# ==============================================================================
# 14. RULE-SET STABILITY — 5 seeds, rule-set overlap
# ==============================================================================
# results/seed_stability.csv is produced by scripts/run_seed_stability.py.
#
# BOTH checkpoints are reported per seed because they answer different questions:
#   * Global Best — the iteration where validation accuracy peaks. Which
#     iteration this is depends on the seed and may lie early in the pruning
#     chain; the rule count then stays close to H_init.
#   * Compact — the final pruning iteration. The recorded schedule runs to
#     completion on every seed, so all seeds are compared at the SAME sparsity
#     level; this is the operating point the paper reports as the interpretable
#     model.
#
# rule_jaccard: Jaccard overlap of rules with the same feature set between two
# runs. feature_jaccard: overlap of the features used by at least one rule.
# The gap between the two separates "same variables chosen, different
# conjunctions" from genuine agreement.
stab = pd.read_csv(require_artifact(
    RESULTS_DIR / "seed_stability.csv",
    "python scripts/run_seed_stability.py --datasets compas german --force",
))

DISPLAY = {"compas": "COMPAS", "german": "German Credit"}

lines = [
    r"\begin{table*}[t]",
    r"\centering",
    r"\caption{Rule-set stability over five seeds (means over the 10 seed pairs). "
    r"\emph{Global Best} is the validation-selected iteration; \emph{Compact} runs the full "
    r"schedule. \emph{Rule set} and \emph{Features} are Jaccard overlaps over rule signatures "
    r"and over the features used by at least one rule; \emph{Links} is the surviving weight "
    r"count.}",
    r"\label{tab:seed_stability}",
    r"\setlength{\tabcolsep}{6pt}",
    r"\renewcommand{\arraystretch}{1.05}",
    r"\begin{tabular}{l l c c c c c}",
    r"\toprule",
    r"\textbf{Dataset} & \textbf{Checkpoint} & \textbf{Rules ($H^*$/$H_0$)} & "
    r"\textbf{Links} & \textbf{Rule set} & \textbf{Features} & \textbf{Acc (\%)} \\",
    r"\midrule",
]

previous = None
CHECKPOINT_ORDER = {"Global Best": 0, "Compact": 1}
for _, row in stab.assign(
    _order=stab["checkpoint"].map(CHECKPOINT_ORDER)
).sort_values(["dataset", "_order"]).iterrows():
    name = DISPLAY.get(row["dataset"], row["dataset"])
    dataset_cell = (
        rf"\multirow{{2}}{{*}}{{{name}}}" if name != previous else ""
    )
    previous = name
    lines.append(
        f"{dataset_cell} & {row['checkpoint']} & "
        f"{row['rules_mean']:.1f}$\\pm${row['rules_std']:.1f} / {int(row['H_init'])} & "
        f"{int(row['links_min'])}--{int(row['links_max'])} & "
        f"{row['rule_jaccard_mean']:.2f}$\\pm${row['rule_jaccard_std']:.2f} & "
        f"{row['feature_jaccard_mean']:.2f}$\\pm${row['feature_jaccard_std']:.2f} & "
        f"{row['accuracy_mean']:.2f}$\\pm${row['accuracy_std']:.2f} \\\\"
    )
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table*}"]
write_table("tab_seed_stability", lines, "rule-set stability across seeds")


% tab_seed_stability — rule-set stability across seeds
\begin{table*}[t]
\centering
\caption{Rule-set stability over five seeds (means over the 10 seed pairs). \emph{Global Best} is the validation-selected iteration; \emph{Compact} runs the full schedule. \emph{Rule set} and \emph{Features} are Jaccard overlaps over rule signatures and over the features used by at least one rule; \emph{Links} is the surviving weight count.}
\label{tab:seed_stability}
\setlength{\tabcolsep}{6pt}
\renewcommand{\arraystretch}{1.05}
\begin{tabular}{l l c c c c c}
\toprule
\textbf{Dataset} & \textbf{Checkpoint} & \textbf{Rules ($H^*$/$H_0$)} & \textbf{Links} & \textbf{Rule set} & \textbf{Features} & \textbf{Acc (\%)} \\
\midrule
\multirow{2}{*}{COMPAS} & Global Best & 31.8$\pm$0.4 / 32 & 266--414 & 0.13$\pm$0.12 & 1.00$\pm$0.00 & 67.14$\pm$0.41 \\
 & Compact & 5.0$\pm$0.7 / 32 & 5--6 & 0.38$\pm$0.15 & 0.40$\pm$0.18 & 66.03$\pm$0.56 \\
\multirow{2}{*}{German Credit} & Global Best & 52.4$\pm$4.0 / 64 & 134--2

PosixPath('/home/ahmet/Projects/ScoreNetSingle/results/tables/tab_seed_stability.tex')